<a href="https://colab.research.google.com/github/umermurtazajadoon-hub/ai-task/blob/main/AiLabProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# AI-Based Network Pathfinding & Attack Simulation System
# COMSATS University Islamabad - CSC 262 Artificial Intelligence
# Instructor: Zeenat Zulfiqar

import heapq
import time
import math
from collections import deque

# ============================================================
# NETWORK TOPOLOGY - 10 Nodes
# ============================================================
# Nodes represent: Attacker(0), Workstations(1-3),
#                  Firewalls(4-5), Servers(6-7),
#                  Database(8), Target(9)

NODE_LABELS = {
    0: "Attacker Entry",
    1: "Workstation-A",
    2: "Workstation-B",
    3: "Workstation-C",
    4: "Firewall-1",
    5: "Firewall-2",
    6: "Web Server",
    7: "App Server",
    8: "Internal DB",
    9: "TARGET (Critical DB)"
}

# Weighted edges: (node1, node2, cost) - cost = vulnerability/risk
EDGES = [
    (0, 1, 2), (0, 2, 4), (0, 3, 6),
    (1, 4, 3), (2, 4, 2), (3, 5, 3),
    (4, 6, 5), (4, 7, 7), (5, 7, 4),
    (6, 8, 3), (7, 8, 2), (7, 9, 8),
    (8, 9, 3), (1, 2, 1), (5, 6, 6),
    (3, 1, 2)
]

START_NODE = 0   # Attacker Entry Point
GOAL_NODE  = 9   # Target Critical Database

# ============================================================
# Build adjacency list
# ============================================================
def build_graph(edges):
    graph = {}
    for u, v, w in edges:
        graph.setdefault(u, []).append((v, w))
        graph.setdefault(v, []).append((u, w))
    return graph

GRAPH = build_graph(EDGES)

# ============================================================
# HEURISTIC for A* - straight-line distance approximation
# (nodes assigned (x,y) positions representing network topology)
# ============================================================
NODE_POSITIONS = {
    0: (0, 5),   1: (2, 7),  2: (2, 5),  3: (2, 3),
    4: (4, 7),   5: (4, 3),  6: (6, 7),  7: (6, 4),
    8: (8, 6),   9: (10, 5)
}

def heuristic(node, goal=GOAL_NODE):
    """Euclidean distance heuristic - estimates cost to reach goal."""
    x1, y1 = NODE_POSITIONS[node]
    x2, y2 = NODE_POSITIONS[goal]
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)


# ============================================================
# 1. BFS - Breadth First Search (Uninformed)
# ============================================================
def bfs(graph, start, goal):
    t0 = time.perf_counter()
    visited = set()
    queue = deque([(start, [start], 0)])
    nodes_expanded = 0

    while queue:
        node, path, cost = queue.popleft()
        if node in visited:
            continue
        visited.add(node)
        nodes_expanded += 1

        if node == goal:
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                queue.append((neighbor, path + [neighbor], cost + weight))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


# ============================================================
# 2. DFS - Depth First Search (Uninformed)
# ============================================================
def dfs(graph, start, goal):
    t0 = time.perf_counter()
    visited = set()
    stack = [(start, [start], 0)]
    nodes_expanded = 0

    while stack:
        node, path, cost = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        nodes_expanded += 1

        if node == goal:
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor], cost + weight))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


# ============================================================
# 3. UCS - Uniform Cost Search (Informed by cost)
# ============================================================
def ucs(graph, start, goal):
    t0 = time.perf_counter()
    heap = [(0, start, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        cost, node, path = heapq.heappop(heap)
        if node in visited and visited[node] <= cost:
            continue
        visited[node] = cost
        nodes_expanded += 1

        if node == goal:
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, weight in graph.get(node, []):
            new_cost = cost + weight
            if neighbor not in visited or visited[neighbor] > new_cost:
                heapq.heappush(heap, (new_cost, neighbor, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


# ============================================================
# 4. A* Search (Informed - uses heuristic)
# ============================================================
def astar(graph, start, goal):
    t0 = time.perf_counter()
    heap = [(heuristic(start), 0, start, [start])]
    visited = {}
    nodes_expanded = 0

    while heap:
        f, g, node, path = heapq.heappop(heap)
        if node in visited:
            continue
        visited[node] = g
        nodes_expanded += 1

        if node == goal:
            return path, g, nodes_expanded, (time.perf_counter() - t0) * 1000

        for neighbor, weight in graph.get(node, []):
            if neighbor not in visited:
                new_g = g + weight
                new_f = new_g + heuristic(neighbor, goal)
                heapq.heappush(heap, (new_f, new_g, neighbor, path + [neighbor]))

    return None, float('inf'), nodes_expanded, (time.perf_counter() - t0) * 1000


# ============================================================
# 5. Hill Climbing (Local Search)
# ============================================================
def hill_climbing(graph, start, goal):
    """
    LOCAL MAXIMA DEMO: Node 6 is a local minimum (lowest h-value
    among neighbors at that point) but NOT the goal. Algorithm
    gets stuck if no neighbor has better h-value.
    """
    t0 = time.perf_counter()
    current = start
    path = [start]
    cost = 0
    nodes_expanded = 0
    visited = {start}

    while current != goal:
        nodes_expanded += 1
        neighbors = [
            (neighbor, weight)
            for neighbor, weight in graph.get(current, [])
            if neighbor not in visited
        ]

        if not neighbors:
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000, "STUCK (Local Maxima!)"

        # Pick neighbor with lowest heuristic value
        best = min(neighbors, key=lambda x: heuristic(x[0], goal))
        best_node, best_weight = best

        # Check if we're moving in right direction
        if heuristic(best_node, goal) >= heuristic(current, goal):
            return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000, "STUCK (Local Maxima!)"

        visited.add(best_node)
        path.append(best_node)
        cost += best_weight
        current = best_node

    return path, cost, nodes_expanded, (time.perf_counter() - t0) * 1000, "SUCCESS"


# ============================================================
# 6. MINIMAX with Alpha-Beta Pruning
# ============================================================
class AttackDefenseGame:
    """
    Two-player adversarial model:
    - Attacker (Maximizer): wants to reach TARGET (node 9)
    - Defender (Minimizer): tries to block paths
    Defender can 'secure' one node per turn (raise its cost to 999)
    """
    def __init__(self, graph, start, goal, depth=3):
        self.graph = graph
        self.start = start
        self.goal = goal
        self.depth = depth
        self.pruning_count = 0

    def evaluate(self, attacker_pos, secured_nodes):
        """Score = negative heuristic (closer to goal = higher score for attacker)"""
        if attacker_pos == self.goal:
            return 100
        if attacker_pos in secured_nodes:
            return -50
        return -heuristic(attacker_pos, self.goal) * 10

    def minimax(self, pos, depth, is_maximizer, secured, alpha, beta, use_pruning=True):
        self.pruning_count += (1 if use_pruning else 0)

        if depth == 0 or pos == self.goal:
            return self.evaluate(pos, secured), pos

        neighbors = [n for n, w in self.graph.get(pos, []) if n not in secured]
        if not neighbors:
            return self.evaluate(pos, secured), pos

        if is_maximizer:
            max_val = float('-inf')
            best_move = pos
            for neighbor in neighbors:
                val, _ = self.minimax(neighbor, depth - 1, False, secured, alpha, beta, use_pruning)
                if val > max_val:
                    max_val, best_move = val, neighbor
                if use_pruning:
                    alpha = max(alpha, val)
                    if beta <= alpha:
                        break
            return max_val, best_move
        else:
            min_val = float('inf')
            best_block = None
            # Defender picks best node to secure
            all_nodes = list(self.graph.keys())
            unsecured = [n for n in all_nodes if n not in secured and n != pos]
            if not unsecured:
                return self.evaluate(pos, secured), pos
            for block_node in unsecured[:5]:  # limit for performance
                new_secured = secured | {block_node}
                val, _ = self.minimax(pos, depth - 1, True, new_secured, alpha, beta, use_pruning)
                if val < min_val:
                    min_val, best_block = val, block_node
                if use_pruning:
                    beta = min(beta, val)
                    if beta <= alpha:
                        break
            return min_val, best_block

    def run_minimax(self):
        t0 = time.perf_counter()
        self.pruning_count = 0
        val, move = self.minimax(self.start, self.depth, True, frozenset(), float('-inf'), float('inf'), False)
        t_minimax = (time.perf_counter() - t0) * 1000

        t0 = time.perf_counter()
        self.pruning_count = 0
        val_ab, move_ab = self.minimax(self.start, self.depth, True, frozenset(), float('-inf'), float('inf'), True)
        pruned = self.pruning_count
        t_ab = (time.perf_counter() - t0) * 1000

        return {
            "minimax_move": move, "minimax_val": val, "minimax_time": t_minimax,
            "ab_move": move_ab, "ab_val": val_ab, "ab_time": t_ab,
            "nodes_pruned": pruned,
            "speedup": t_minimax / t_ab if t_ab > 0 else 1
        }


# ============================================================
# RUN ALL ALGORITHMS & DISPLAY RESULTS
# ============================================================
def path_to_names(path):
    return " → ".join(NODE_LABELS[n] for n in path)

def run_all():
    print("=" * 70)
    print("  AI-BASED NETWORK PATHFINDING & ATTACK SIMULATION SYSTEM")
    print("  COMSATS University Islamabad | CSC 262 | Zeenat Zulfiqar")
    print("=" * 70)
    print(f"\n  Start: {NODE_LABELS[START_NODE]}  →  Goal: {NODE_LABELS[GOAL_NODE]}\n")

    results = {}

    # BFS
    path, cost, nodes_exp, t = bfs(GRAPH, START_NODE, GOAL_NODE)
    results['BFS'] = (path, cost, nodes_exp, t)
    print(f"[BFS]  Path: {path}")
    print(f"       Names: {path_to_names(path)}")
    print(f"       Cost: {cost} | Nodes Expanded: {nodes_exp} | Time: {t:.4f} ms\n")

    # DFS
    path, cost, nodes_exp, t = dfs(GRAPH, START_NODE, GOAL_NODE)
    results['DFS'] = (path, cost, nodes_exp, t)
    print(f"[DFS]  Path: {path}")
    print(f"       Names: {path_to_names(path)}")
    print(f"       Cost: {cost} | Nodes Expanded: {nodes_exp} | Time: {t:.4f} ms\n")

    # UCS
    path, cost, nodes_exp, t = ucs(GRAPH, START_NODE, GOAL_NODE)
    results['UCS'] = (path, cost, nodes_exp, t)
    print(f"[UCS]  Path: {path}")
    print(f"       Names: {path_to_names(path)}")
    print(f"       Cost: {cost} | Nodes Expanded: {nodes_exp} | Time: {t:.4f} ms\n")

    # A*
    path, cost, nodes_exp, t = astar(GRAPH, START_NODE, GOAL_NODE)
    results['A*'] = (path, cost, nodes_exp, t)
    print(f"[A*]   Path: {path}")
    print(f"       Names: {path_to_names(path)}")
    print(f"       Cost: {cost} | Nodes Expanded: {nodes_exp} | Time: {t:.4f} ms\n")

    # Hill Climbing
    path, cost, nodes_exp, t, status = hill_climbing(GRAPH, START_NODE, GOAL_NODE)
    results['Hill'] = (path, cost, nodes_exp, t)
    print(f"[Hill Climbing] Status: {status}")
    print(f"       Path: {path}")
    print(f"       Names: {path_to_names(path)}")
    print(f"       Cost: {cost} | Nodes Expanded: {nodes_exp} | Time: {t:.4f} ms\n")

    # Minimax + Alpha-Beta
    game = AttackDefenseGame(GRAPH, START_NODE, GOAL_NODE, depth=3)
    mm_results = game.run_minimax()
    print(f"[Minimax]        Best Move from start: {NODE_LABELS[mm_results['minimax_move']]} | Time: {mm_results['minimax_time']:.4f} ms")
    print(f"[Alpha-Beta]     Best Move from start: {NODE_LABELS[mm_results['ab_move']]} | Time: {mm_results['ab_time']:.4f} ms")
    print(f"                 Nodes pruned: {mm_results['nodes_pruned']} | Speedup: {mm_results['speedup']:.2f}x\n")

    # Comparison Table
    print("=" * 70)
    print(f"{'Algorithm':<15} {'Path':<25} {'Total Cost':<12} {'Nodes Exp.':<14} {'Time (ms)':<10}")
    print("-" * 70)
    for algo, (path, cost, nodes_exp, t) in results.items():
        path_str = str(path) if path else "None"
        print(f"{algo:<15} {path_str:<25} {cost:<12} {nodes_exp:<14} {t:<10.4f}")
    print("=" * 70)

run_all()


  AI-BASED NETWORK PATHFINDING & ATTACK SIMULATION SYSTEM
  COMSATS University Islamabad | CSC 262 | Zeenat Zulfiqar

  Start: Attacker Entry  →  Goal: TARGET (Critical DB)

[BFS]  Path: [0, 1, 4, 7, 9]
       Names: Attacker Entry → Workstation-A → Firewall-1 → App Server → TARGET (Critical DB)
       Cost: 20 | Nodes Expanded: 10 | Time: 0.0288 ms

[DFS]  Path: [0, 3, 1, 2, 4, 7, 9]
       Names: Attacker Entry → Workstation-C → Workstation-A → Workstation-B → Firewall-1 → App Server → TARGET (Critical DB)
       Cost: 26 | Nodes Expanded: 7 | Time: 0.0099 ms

[UCS]  Path: [0, 1, 2, 4, 6, 8, 9]
       Names: Attacker Entry → Workstation-A → Workstation-B → Firewall-1 → Web Server → Internal DB → TARGET (Critical DB)
       Cost: 16 | Nodes Expanded: 10 | Time: 0.0217 ms

[A*]   Path: [0, 1, 2, 4, 6, 8, 9]
       Names: Attacker Entry → Workstation-A → Workstation-B → Firewall-1 → Web Server → Internal DB → TARGET (Critical DB)
       Cost: 16 | Nodes Expanded: 10 | Time: 0.0270 ms

[